# Sunflower Plant Biomechanics — GPU-Accelerated Simulation

**v2 — proper cantilever bending**: rigidity now drops with height via a per-node lateral restoring force whose stiffness follows `k(h) = k_base · (1 − h/H)ⁿ`. Base is rigid, tip is floppy — plant curves like a cantilever instead of tilting rigidly.

| Cell | Role |
|---|---|
| 1 | Imports + Warp GPU init |
| 2 | `CONFIG` — physical constants + stiffness profile |
| 3 | `build_plant()` — nodes, axial & bending springs, per-node lateral k |
| 4 | GPU kernels — gravity, drag, rain, springs, lateral restore, integrate |
| 5 | `run_simulation()` — time loop with persistent GPU buffers |
| 6 | `plot_snapshots()` — 6-panel figure |
| 7 | Execute |

**Physics**: Euler-Bernoulli axial springs (`k = EA/L`), lateral flexural anchoring `F = -k(h)·Δx` with steep `(1-h/H)⁵` falloff (256× tapered EI via `I ∝ r⁴` reinforces this), quadratic wind drag (`½ρCdAv²`), rain momentum flux (`ρRvA`), semi-implicit Euler, 40 substeps for stability.


## 1. Imports & Warp GPU init

In [ ]:
import warp as wp
import numpy as np
import matplotlib.pyplot as plt

wp.init()
print("Device:", wp.get_device())


## 2. Physical configuration

In [ ]:
# ─── Geometry (young sunflower) ──────────────────────────────────────
STEM_NODES   = 12
STEM_HEIGHT  = 1.0
L0_STEM      = STEM_HEIGHT / (STEM_NODES - 1)
R_BASE       = 0.008
R_TIP        = 0.002

# ─── Stem material ───────────────────────────────────────────────────
E_STEM       = 5.0e7          # 50 MPa
RHO_STEM     = 700.0
CD_STEM      = 1.0

# ─── Leaves ──────────────────────────────────────────────────────────
LEAF_CONFIG  = [
    (3,  55, 3), (3, -55, 3),
    (6,  60, 4), (6, -60, 4),
    (9,  50, 3), (9, -50, 3),
]
L0_LEAF      = 0.06
R_LEAF       = 0.0015
A_LEAF_BLADE = 0.004
E_LEAF       = 1.0e7
RHO_LEAF     = 500.0
CD_LEAF      = 2.0

# ─── Environment ─────────────────────────────────────────────────────
G            = 9.81
RHO_AIR      = 1.225
RHO_WATER    = 1000.0
V_RAIN       = 9.0
RAIN_RATE    = 5.0e-5

# ─── Ground / soil ───────────────────────────────────────────────────
SOIL_DEPTH   = 0.05
ROOT_DAMPING = 0.55

# ─── Integration ─────────────────────────────────────────────────────
DT           = 1.0 / 60.0
SUBSTEPS     = 40
DAMPING      = 0.985

# ─── Flexural rigidity profile (THIS is what makes base rigid, tip floppy) ──
# Per-node lateral restoring stiffness:   k(h) = K_BASE·(1 - h/H)^p + K_FLOOR
# Strong at soil (very rigid) → near zero at tip (free to whip in wind).
K_LATERAL_BASE  = 4000.0      # N/m — very stiff at ground
K_LATERAL_FLOOR = 0.2         # N/m — near-free at tip
LATERAL_POWER   = 5.0         # exponent — bigger = sharper drop-off
LEAF_LATERAL_K  = 0.1         # leaves are always very soft

# ─── Scenario timeline ───────────────────────────────────────────────
TOTAL_FRAMES = 150
WIND_CALM    = 0.5
WIND_STRONG  = 10.0           # stronger wind (was 8.0) to see curvature
RAIN_START   = 50
RAIN_END     = 100

def wind_speed(frame: int) -> float:
    if frame < RAIN_START:         return WIND_STRONG
    if frame < RAIN_END:           return WIND_STRONG * 0.75
    if frame < RAIN_END + 20:      return 2.0
    return WIND_CALM

print(f"sub-dt = {DT/SUBSTEPS*1000:.3f} ms  |  k_base = {K_LATERAL_BASE} N/m, k_tip ≈ {K_LATERAL_FLOOR} N/m")


## 3. Plant geometry + per-node lateral stiffness

Crucial piece: each stem node gets a **lateral anchoring stiffness** `k_lat(h)` that falls off sharply with height. Physically this represents the cumulative flexural rigidity `EI(h)` of the stem below node `h`, pulling the node back toward its rest column.


In [ ]:
def stem_radius(i, n=STEM_NODES):
    t = i / (n - 1)
    return R_BASE + (R_TIP - R_BASE) * t

def area(r):           return np.pi * r * r
def second_moment(r):  return np.pi * r**4 / 4.0
def k_axial(E, r, L):  return E * area(r) / L
def node_mass_half(rho, r, L):  return 0.5 * rho * area(r) * L

def lateral_stiffness(h, H=STEM_HEIGHT):
    f = max(0.0, 1.0 - h/H)
    return K_LATERAL_BASE * (f ** LATERAL_POWER) + K_LATERAL_FLOOR

def build_plant():
    pos, mass, is_root, is_leaf, front_area, k_lat_list = [], [], [], [], [], []
    parents, children, L0_list, k_ax_list = [], [], [], []

    # ── Stem ────────────────────────────────────────────────
    for i in range(STEM_NODES):
        y = i * L0_STEM
        pos.append([0.0, y, 0.0])
        is_root.append(1 if i == 0 else 0)
        is_leaf.append(0)
        r_here = stem_radius(i)

        m_i = 0.0
        if i > 0:              m_i += node_mass_half(RHO_STEM, stem_radius(i-1), L0_STEM)
        if i < STEM_NODES - 1: m_i += node_mass_half(RHO_STEM, stem_radius(i),   L0_STEM)
        mass.append(max(m_i, 1e-6))
        front_area.append(2.0 * r_here * L0_STEM)
        k_lat_list.append(lateral_stiffness(y))

        if i > 0:
            r_edge = 0.5 * (stem_radius(i-1) + stem_radius(i))
            parents.append(i - 1)
            children.append(i)
            L0_list.append(L0_STEM)
            k_ax_list.append(k_axial(E_STEM, r_edge, L0_STEM))

    # ── Leaves (always soft; flutter freely) ────────────────
    for (attach, angle_deg, n_seg) in LEAF_CONFIG:
        a = np.radians(angle_deg)
        dx, dy = np.sin(a) * L0_LEAF, np.cos(a) * L0_LEAF
        base = pos[attach]
        prev = attach
        for j in range(1, n_seg + 1):
            idx = len(pos)
            pos.append([base[0] + dx*j, base[1] + dy*j, 0.0])
            is_root.append(0); is_leaf.append(1)
            mass.append(node_mass_half(RHO_LEAF, R_LEAF, L0_LEAF) * 2.0)
            front_area.append(A_LEAF_BLADE)
            k_lat_list.append(LEAF_LATERAL_K)

            parents.append(prev); children.append(idx)
            L0_list.append(L0_LEAF)
            k_ax_list.append(k_axial(E_LEAF, R_LEAF, L0_LEAF))
            prev = idx

    pos_np = np.asarray(pos, dtype=np.float32)
    return dict(
        pos          = pos_np,
        rest_pos     = pos_np.copy(),
        vel          = np.zeros_like(pos_np),
        mass         = np.asarray(mass,   dtype=np.float32),
        is_root      = np.asarray(is_root, dtype=np.int32),
        is_leaf      = np.asarray(is_leaf, dtype=np.int32),
        frontal_area = np.asarray(front_area, dtype=np.float32),
        k_lateral    = np.asarray(k_lat_list, dtype=np.float32),
        parents      = np.asarray(parents,  dtype=np.int32),
        children     = np.asarray(children, dtype=np.int32),
        L0           = np.asarray(L0_list,  dtype=np.float32),
        k_axial      = np.asarray(k_ax_list, dtype=np.float32),
    )

_probe = build_plant()
print(f"Nodes: {len(_probe['pos'])}   axial springs: {len(_probe['L0'])}")
print("Stem k_lateral profile (base → tip):")
for i in range(STEM_NODES):
    print(f"  node {i:2d}  y={i*L0_STEM:.3f} m   k = {_probe['k_lateral'][i]:8.2f} N/m")


## 4. GPU physics kernels

Added `k_lateral_restore` — this is the kernel that creates the cantilever behaviour. Each node feels a spring pulling it back to its rest X/Z column, with per-node stiffness from `k_lateral`.

In [ ]:
@wp.kernel
def k_zero_forces(forces: wp.array(dtype=wp.vec3)):
    i = wp.tid()
    forces[i] = wp.vec3(0.0, 0.0, 0.0)

@wp.kernel
def k_gravity(forces: wp.array(dtype=wp.vec3),
              mass:   wp.array(dtype=wp.float32),
              g: float):
    i = wp.tid()
    forces[i] = forces[i] + wp.vec3(0.0, -g * mass[i], 0.0)

@wp.kernel
def k_wind_drag(positions:  wp.array(dtype=wp.vec3),
                velocities: wp.array(dtype=wp.vec3),
                forces:     wp.array(dtype=wp.vec3),
                area_arr:   wp.array(dtype=wp.float32),
                is_leaf:    wp.array(dtype=wp.int32),
                wind_x: float,
                rho_air: float, cd_stem: float, cd_leaf: float):
    i = wp.tid()
    v_rel = wp.vec3(wind_x, 0.0, 0.0) - velocities[i]
    speed = wp.length(v_rel)
    cd = cd_stem
    if is_leaf[i] == 1:
        cd = cd_leaf
    h_factor = positions[i][1] * 0.5 + 0.5
    mag = 0.5 * rho_air * cd * area_arr[i] * speed * h_factor
    forces[i] = forces[i] + v_rel * mag

@wp.kernel
def k_rain(forces:   wp.array(dtype=wp.vec3),
           area_arr: wp.array(dtype=wp.float32),
           is_leaf:  wp.array(dtype=wp.int32),
           rain_on:  int,
           rho_w: float, rain_rate: float, v_rain: float):
    i = wp.tid()
    if rain_on == 0:
        return
    f = rho_w * rain_rate * v_rain * area_arr[i]
    if is_leaf[i] == 1:
        f = f * 25.0
    else:
        f = f * 3.0
    forces[i] = forces[i] + wp.vec3(0.3 * f, -f, 0.0)

@wp.kernel
def k_axial_springs(positions:  wp.array(dtype=wp.vec3),
                    velocities: wp.array(dtype=wp.vec3),
                    forces:     wp.array(dtype=wp.vec3),
                    parents:    wp.array(dtype=wp.int32),
                    children:   wp.array(dtype=wp.int32),
                    L0:         wp.array(dtype=wp.float32),
                    k_arr:      wp.array(dtype=wp.float32),
                    c_damp: float):
    e = wp.tid()
    p = parents[e]; c = children[e]
    diff = positions[c] - positions[p]
    dist = wp.length(diff)
    if dist > 1e-8:
        n = diff / dist
        ext = dist - L0[e]
        v_rel = wp.dot(velocities[c] - velocities[p], n)
        f_mag = k_arr[e] * ext + c_damp * v_rel
        force = n * f_mag
        wp.atomic_add(forces, p,  force)
        wp.atomic_add(forces, c, -force)

@wp.kernel
def k_lateral_restore(positions: wp.array(dtype=wp.vec3),
                      rest_pos:  wp.array(dtype=wp.vec3),
                      forces:    wp.array(dtype=wp.vec3),
                      k_lat:     wp.array(dtype=wp.float32),
                      is_root:   wp.array(dtype=wp.int32)):
    # Flexural anchoring: pull each node back toward its rest X/Z column.
    # Stiffness k_lat[i] is set per-node — large at base, ~0 at tip.
    i = wp.tid()
    if is_root[i] == 1:
        return
    dx = rest_pos[i][0] - positions[i][0]
    dz = rest_pos[i][2] - positions[i][2]
    forces[i] = forces[i] + wp.vec3(dx * k_lat[i], 0.0, dz * k_lat[i])

@wp.kernel
def k_integrate(positions:  wp.array(dtype=wp.vec3),
                velocities: wp.array(dtype=wp.vec3),
                forces:     wp.array(dtype=wp.vec3),
                mass:       wp.array(dtype=wp.float32),
                is_root:    wp.array(dtype=wp.int32),
                damping: float, soil_damp: float, soil_y: float,
                dt: float):
    i = wp.tid()
    if is_root[i] == 1:
        velocities[i] = wp.vec3(0.0, 0.0, 0.0)
        return
    a = forces[i] / mass[i]
    a_mag = wp.length(a)
    a_max = float(500.0)
    if a_mag > a_max:
        a = a * (a_max / a_mag)
    d = damping
    if positions[i][1] < soil_y:
        d = soil_damp
    velocities[i] = (velocities[i] + a * dt) * d
    positions[i]  = positions[i]  + velocities[i] * dt
    if positions[i][1] < 0.0:
        positions[i]  = wp.vec3(positions[i][0], 0.0, positions[i][2])
        velocities[i] = wp.vec3(velocities[i][0], 0.0, velocities[i][2])

print("Kernels compiled.")


## 5. Simulation loop

In [ ]:
SNAPSHOT_FRAMES = [20, 49, 60, 85, 100, 140]

def _gpu(arr, dtype):
    return wp.array(arr, dtype=dtype, device="cuda")

def run_simulation(plant, total_frames=TOTAL_FRAMES, substeps=SUBSTEPS, verbose=True):
    n       = len(plant["pos"])
    n_edges = len(plant["L0"])

    pos      = _gpu(plant["pos"],          wp.vec3)
    restpos  = _gpu(plant["rest_pos"],     wp.vec3)
    vel      = _gpu(plant["vel"],          wp.vec3)
    frc      = wp.zeros(n, dtype=wp.vec3,  device="cuda")
    mass     = _gpu(plant["mass"],         wp.float32)
    root     = _gpu(plant["is_root"],      wp.int32)
    leaf     = _gpu(plant["is_leaf"],      wp.int32)
    areaG    = _gpu(plant["frontal_area"], wp.float32)
    kLatG    = _gpu(plant["k_lateral"],    wp.float32)

    par      = _gpu(plant["parents"],  wp.int32)
    chd      = _gpu(plant["children"], wp.int32)
    L0a      = _gpu(plant["L0"],       wp.float32)
    kA       = _gpu(plant["k_axial"],  wp.float32)

    c_damp   = 0.15
    sub_dt   = DT / substeps
    snapshots = {}
    save_set  = set(SNAPSHOT_FRAMES)

    for frame in range(total_frames):
        w = wind_speed(frame)
        rain_on = 1 if RAIN_START <= frame < RAIN_END else 0

        for _ in range(substeps):
            wp.launch(k_zero_forces,   dim=n,       inputs=[frc])
            wp.launch(k_gravity,       dim=n,       inputs=[frc, mass, G])
            wp.launch(k_wind_drag,     dim=n,
                      inputs=[pos, vel, frc, areaG, leaf, w,
                              RHO_AIR, CD_STEM, CD_LEAF])
            wp.launch(k_rain,          dim=n,
                      inputs=[frc, areaG, leaf, rain_on,
                              RHO_WATER, RAIN_RATE, V_RAIN])
            if n_edges > 0:
                wp.launch(k_axial_springs, dim=n_edges,
                          inputs=[pos, vel, frc, par, chd, L0a, kA, c_damp])
            wp.launch(k_lateral_restore, dim=n,
                      inputs=[pos, restpos, frc, kLatG, root])
            wp.launch(k_integrate, dim=n,
                      inputs=[pos, vel, frc, mass, root,
                              DAMPING, ROOT_DAMPING, SOIL_DEPTH, sub_dt])

        if frame in save_set:
            snapshots[frame] = pos.numpy().copy()

    if verbose:
        print(f"Simulated {total_frames} frames × {substeps} substeps "
              f"({total_frames*substeps} physics steps) on GPU.")
    return snapshots


## 6. Visualization — 6-panel timeline

In [ ]:
SNAPSHOT_LABELS = {
    20:  "Wind builds\n(t=0.33 s)",
    49:  "Full wind\n(t=0.82 s)",
    60:  "Rain starts\n(t=1.00 s)",
    85:  "Heavy rain\n(t=1.42 s)",
    100: "Rain stops\n(t=1.67 s)",
    140: "Recovering\n(t=2.33 s)",
}

def plot_snapshots(snapshots, plant, save_path="plant_physics_accurate.png"):
    frames = sorted(snapshots.keys())
    fig, axes = plt.subplots(1, len(frames), figsize=(3 * len(frames), 5))

    stem_palette = ['#1D9E75','#0F6E56','#085041','#04342C','#7EBDA8','#5DCAA5']
    leaf_color = '#5DCAA5'
    rain_col   = '#378ADD'

    for col, f in enumerate(frames):
        ax = axes[col]
        snap = snapshots[f]
        rain_flag = RAIN_START <= f < RAIN_END

        ax.axhspan(-0.3, 0.0, color='#8B5E3C', alpha=0.22)
        ax.axhline(y=0, color='#8B5E3C', linewidth=2.5, alpha=0.9)

        if rain_flag:
            rng = np.random.default_rng(seed=col * 7)
            for _ in range(24):
                rx = rng.uniform(-0.5, 1.2)
                ry = rng.uniform(0.15, 1.30)
                ax.annotate('', xy=(rx + 0.04, ry - 0.12), xytext=(rx, ry),
                            arrowprops=dict(arrowstyle='->',
                                            color=rain_col, lw=0.9, alpha=0.5))
            ax.text(0.97, 0.97, 'RAIN', transform=ax.transAxes,
                    fontsize=8, color=rain_col, ha='right', va='top',
                    fontweight='bold')

        ax.plot(plant["rest_pos"][:STEM_NODES, 0],
                plant["rest_pos"][:STEM_NODES, 1],
                '--', color='gray', linewidth=1.0, alpha=0.35, zorder=1)

        ax.plot(snap[:STEM_NODES, 0], snap[:STEM_NODES, 1],
                'o-', color=stem_palette[col % len(stem_palette)],
                linewidth=2.8, markersize=4.5, zorder=3)

        cursor = STEM_NODES
        for (attach, ang, n_seg) in LEAF_CONFIG:
            idx = [attach] + list(range(cursor, cursor + n_seg))
            ax.plot(snap[idx, 0], snap[idx, 1],
                    'o-', color=leaf_color, linewidth=1.5,
                    markersize=3, alpha=0.9, zorder=2)
            cursor += n_seg

        ax.plot(0, 0, 'o', color='#8B5E3C', markersize=9, zorder=5)
        ax.set_title(SNAPSHOT_LABELS.get(f, f"frame {f}"), fontsize=9)
        ax.set_xlim(-0.6, 1.4)
        ax.set_ylim(-0.2, 1.35)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.2)

    plt.suptitle(
        "Sunflower — cantilever bending (rigid base, floppy tip) under wind + rain  |  GPU: Warp on RTX 4070",
        fontsize=10,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return save_path


## 7. Run

In [ ]:
plant = build_plant()
snapshots = run_simulation(plant)
path = plot_snapshots(snapshots, plant)
print("Saved:", path)
